# 03: Publication tables

Builds tables from existing files in `outputs/py/` only. Nothing is re-estimated.
Coefficients come from `estimates.csv`. Descriptives are computed from
`dk_clean.parquet`. D&K values come only from `data/comparisons/dk_published_penalty.csv`.

Output per table, in `outputs/tables/`
HTML (Word), `.tex` (Overleaf, via `as_latex()`), and the underlying numbers as CSV.
PNG export is not in this notebook yet. It needs Selenium, which is not installed (Chrome is present).

In [6]:
import os
import math
import numpy as np
import pandas as pd
from great_tables import GT, loc
from great_tables import style as gt_style

OUT = "../outputs"
os.makedirs(f"{OUT}/tables", exist_ok=True)

estimates = pd.read_csv(f"{OUT}/py/estimates.csv")
bench = pd.read_csv("../data/comparisons/dk_published_penalty.csv")

MODEL_COV = ["log_wage", "outsourced", "union", "parttime", "educ_cat", "female", "black", "hispanic",
             "age", "age2", "msa", "central_city", "cc_unknown", "geo_unidentified"]
COLS = ["occupation", "YEAR", "MONTH", "outsourced", "EARNWT", "ym", "real_wage_trimmed",
        "union", "parttime", "age", "female", "black", "hispanic", "educ_cat"] + MODEL_COV + ["gov_employee"]
COLS = list(dict.fromkeys(COLS))
dk_base = pd.read_parquet(f"{OUT}/py/dk_clean.parquet", columns=COLS)
print("base sample:", dk_base.shape)

base sample: (166391, 21)


In [ ]:
import re
def stars(p):
    return "***" if p < .01 else "**" if p < .05 else "*" if p < .10 else ""

def p_two_sided(z):
    return math.erfc(abs(z) / math.sqrt(2))

def get_row(df, **kw):
    m = np.ones(len(df), dtype=bool)
    for k, v in kw.items():
        m &= (df[k] == v).to_numpy()
    sub = df[m]
    assert len(sub) == 1, (kw, len(sub))
    return sub.iloc[0]

def style(gt_obj):
    return (gt_obj
            .opt_table_font(font=["Times New Roman", "Times", "serif"])
            .tab_options(table_font_size="10pt"))

def add_tex_rules(tex, rule_before):
    """great_tables drops borders in LaTeX. Add a vertical rule to the tabular column spec
    before each data column index in rule_before (0-based, stub excluded)."""
    pat = r"(\\begin\{tabular\*\}\{\\linewidth\}\{@\{\\extracolsep\{\\fill\}\})([^}]*)\}"
    m = re.search(pat, tex)
    assert m, "tabular spec not found"
    spec = m.group(2)
    stub, data = spec[0], spec[1:]
    bar = data.startswith("|")
    data = data.lstrip("|")
    new = "".join(("|" if i in rule_before else "") + ch for i, ch in enumerate(data))
    return tex.replace(m.group(0), m.group(1) + stub + ("|" if bar else "") + new + "}", 1)

def save_outputs(gt_obj, raw, stem, tex_rules=None):
    with open(f"{OUT}/tables/{stem}.html", "w") as f:
        f.write(gt_obj.as_raw_html())
    tex = gt_obj.as_latex()
    if tex_rules:
        tex = add_tex_rules(tex, tex_rules)
    with open(f"{OUT}/tables/{stem}.tex", "w") as f:
        f.write(tex)
    raw.to_csv(f"{OUT}/tables/{stem}.csv", index=False)
    print("saved", stem)

DEF_REPLICATION = "Author's Replication: the author's wage regression (row-1 model, no gov_employee) on 1983-2000."
DEF_EXTENSION = "Author's Extension: the same model on 2001-2025."
DEF_DK = "D&K = Dube and Kaplan's (2010) reported values."
EXTRAP_RULE_DEF = ("The vertical rule separates the Author's Replication (1983-2000) "
                       "from the Author's Extension (2001-2025).")

SOURCE_FE = ("FE: state x year and clustering by survey month (ym) apply to regression tables only; "
             "Tables 1-2 are descriptive. Stars: * p<.10, ** p<.05, *** p<.01, two-tailed.")

def stacked_rows(entries, label_key, est_cols, single_cols, group_key=None):
    """Regression layout: each term is a coefficient row with its SE in parentheses on the row below.
    entries: dicts with `label`, `est` (col -> (estimate, se, p), or None for a blank cell),
    `single` (col -> text, shown on the coefficient row only), and (if group_key) `group`
    (a panel label, e.g. for GT's groupname_col). Each estimate column has a star column right
    of it. Returns display rows and the 0-based positions of the SE rows."""
    rows, se_rows = [], []
    for e in entries:
        coef, se = {label_key: e["label"]}, {label_key: ""}
        if group_key is not None:
            coef[group_key] = se[group_key] = e.get("group", "")
        for col in est_cols:
            cell = e["est"].get(col)
            if cell is None:
                coef[col] = coef[col + "_star"] = se[col] = se[col + "_star"] = ""
            else:
                est, s, p = cell
                coef[col], coef[col + "_star"] = f"{est:.3f}", stars(p)
                se[col], se[col + "_star"] = f"({s:.3f})", ""
        for col in single_cols:
            coef[col], se[col] = e["single"].get(col, ""), ""
        rows.append(coef)
        se_rows.append(len(rows))
        rows.append(se)
    return rows, se_rows

def stacked_frame(rows, label_key, est_cols, single_cols, group_key=None):
    cols = ([group_key] if group_key is not None else []) + [label_key]
    cols += [c for col in est_cols for c in (col, col + "_star")] + single_cols
    return pd.DataFrame(rows, columns=cols)

def style_regression(gt_obj, est_cols, se_rows, star_cols, extrap_cols=None):
    gt_obj = (gt_obj
            .cols_align(align="right", columns=est_cols)
            .cols_align(align="left", columns=star_cols)
            .cols_label(cases={c: "" for c in star_cols})
            .cols_width(cases={c: "22px" for c in star_cols})
            .tab_style(style=gt_style.text(align="center"),
                       locations=loc.body(columns=est_cols, rows=se_rows))
            .tab_style(style=gt_style.css("padding-left: 0"),
                       locations=loc.body(columns=star_cols)))
    if extrap_cols:
        gt_obj = gt_obj.tab_style(
            style=gt_style.borders(sides="left", color="#444444", style="solid", weight="1px"),
            locations=[loc.body(columns=extrap_cols), loc.column_labels(columns=extrap_cols)])
    return gt_obj

In [ ]:
periods5 = ["2001-2005", "2006-2010", "2011-2015", "2016-2019", "2021-2025"]
ranges = {"2001-2005": (2001, 2005), "2006-2010": (2006, 2010), "2011-2015": (2011, 2015),
          "2016-2019": (2016, 2019), "2021-2025": (2021, 2025)}

author_sample = dk_base.dropna(subset=MODEL_COV).copy()
print("author regression sample:", author_sample.shape)

def wmean(x, w):
    x = np.asarray(x, dtype=float); w = np.asarray(w, dtype=float)
    return float(np.sum(w * x) / np.sum(w))

def cell_stats(occ, label, status):
    y0, y1 = ranges[label]
    g = author_sample[(author_sample["occupation"] == occ) & author_sample["YEAR"].between(y0, y1) & (author_sample["outsourced"] == status)]
    w = g["EARNWT"]
    edu = g["educ_cat"].astype(str)
    out = {
        "real_wage_trimmed": wmean(g["real_wage_trimmed"], w),
        "union": wmean(g["union"], w),
        "parttime": wmean(g["parttime"], w),
        "gov_employee": wmean(g["gov_employee"], w),
        "age": wmean(g["age"], w),
        "female": wmean(g["female"], w),
        "black": wmean(g["black"], w),
        "hispanic": wmean(g["hispanic"], w),
        "edu_lths": wmean(edu.isin(["no_school", "primary", "hs_attend"]).astype(float), w),
        "edu_hs": wmean((edu == "hs_complete").astype(float), w),
        "edu_some": wmean((edu == "some_college").astype(float), w),
        "edu_ba": wmean((edu == "college").astype(float), w),
        "n_unweighted": int(len(g)),
    }
    base = dk_base[(dk_base["occupation"] == occ) & dk_base["YEAR"].between(y0, y1)]
    n_months = base["ym"].nunique()
    out["workers_k"] = round(base.loc[base["outsourced"] == status, "EARNWT"].sum() / n_months / 1000)
    return out, n_months

rows_spec = [
    ("Employment and wages", "Mean hourly wage (2025$)", "real_wage_trimmed", "dollar"),
    ("Employment and wages", "Unionization rate", "union", "pct"),
    ("Employment and wages", "Part-time rate", "parttime", "pct"),
    ("Employment and wages", "Government employee", "gov_employee", "pct"),
    ("Demographics", "Mean age", "age", "num"),
    ("Demographics", "Female", "female", "pct"),
    ("Demographics", "Black", "black", "pct"),
    ("Demographics", "Hispanic", "hispanic", "pct"),
    ("Education", "Less than high school", "edu_lths", "pct"),
    ("Education", "High school / GED", "edu_hs", "pct"),
    ("Education", "Some college", "edu_some", "pct"),
    ("Education", "Bachelor's or higher", "edu_ba", "pct"),
    ("Sample", "Unweighted N (regression sample)", "n_unweighted", "int"),
    ("Sample", "Workers represented (thousands)", "workers_k", "int"),
]

def fmt(v, kind):
    if kind == "dollar": return f"${v:,.2f}"
    if kind == "pct": return f"{100 * v:.1f}%"
    if kind == "num": return f"{v:.1f}"
    return f"{int(v):,}"

def workforce_table(occ, occ_label, table_no, letter):
    stats = {}
    for label in periods5:
        for status in (0, 1):
            stats[(label, status)], _ = cell_stats(occ, label, status)
    disp_rows, raw_rows = [], []
    for group, measure, key, kind in rows_spec:
        drow, rrow = {"group": group, "measure": measure}, {"group": group, "measure": measure}
        for label in periods5:
            for status, tag in [(0, "in"), (1, "con")]:
                col = f"{label}__{tag}"
                v = stats[(label, status)][key]
                drow[col] = fmt(v, kind)
                rrow[col] = v
        disp_rows.append(drow)
        raw_rows.append(rrow)
    disp, raw = pd.DataFrame(disp_rows), pd.DataFrame(raw_rows)
    gt_obj = (
        GT(disp, rowname_col="measure", groupname_col="group")
        .tab_header(title=f"Table {table_no}: Workforce Characteristics, {occ_label}, 2001-2025",
                    subtitle="Contractor = Outsourced; In-House = Directly Employed")
        .tab_source_note(source_note=(
            "Data: CPS-ORG via IPUMS. Years: 2001-2025, 2020 excluded. Part-time = usual hours under 30. "
            "Weights: EARNWT. Statistics use the author's regression sample (non-missing log_wage and every model covariate); "
            "N = observations with non-missing wages and all model covariates. Workers represented is average monthly employment estimated from EARNWT "
            "on all sampled base-sample workers, including those excluded for missing wages or covariates: sum(EARNWT) "
            "divided by the number of distinct survey months (ym) for that occupation in the period, in thousands, "
            "rounded to the nearest thousand. " + SOURCE_FE))
    )
    for label in periods5:
        gt_obj = gt_obj.tab_spanner(label=label, columns=[f"{label}__in", f"{label}__con"])
    labels = {}
    for label in periods5:
        labels[f"{label}__in"] = "In-house"
        labels[f"{label}__con"] = "Contractor"
    gt_obj = style(gt_obj.cols_label(**labels))
    return gt_obj, raw

gt1, raw1 = workforce_table("janitor", "Janitors", 1, "a")
save_outputs(gt1, raw1, "table1_janitors_workforce")
gt2, raw2 = workforce_table("guard", "Guards", 2, "b")
save_outputs(gt2, raw2, "table2_guards_workforce")

In [ ]:
# Table A3: D&K pooled 1983-2000 coefficients, Author's Replication vs. D&K reported (row 1)
entries3, raw3 = [], []
for occ, occ_label in [("janitor", "Janitors"), ("guard", "Guards")]:
    m = get_row(estimates, spec="author", model="base", occ=occ, sample="1983-2000", term="outsourced")
    d = get_row(bench, occ=occ_label, row=1, term="outsourced")
    d_p = p_two_sided(d["estimate"] / d["se"])
    entries3.append({"label": occ_label,
                     "est": {"author": (m["estimate"], m["se"], m["p"]),
                             "dk": (d["estimate"], d["se"], d_p)},
                     "single": {"author_n": f"{int(m['n_obs']):,}",
                                "dk_n": f"{int(d['n']):,}"}})
    raw3.append({"occupation": occ_label, "author_estimate": m["estimate"], "author_se": m["se"],
                 "author_p": m["p"], "author_n": int(m["n_obs"]), "dk_estimate": d["estimate"],
                 "dk_se": d["se"], "dk_n": int(d["n"])})
single_cols = ["author_n", "dk_n"]
rows3, se_rows3 = stacked_rows(entries3, "occupation", ["author", "dk"], single_cols)
disp3 = stacked_frame(rows3, "occupation", ["author", "dk"], single_cols)
disp3 = disp3[["occupation", "author", "author_star", "author_n", "dk", "dk_star", "dk_n"]]
raw3 = pd.DataFrame(raw3)
gt3 = style(
    GT(disp3, rowname_col="occupation")
    .tab_header(title="Table A3: D&K Pooled 1983-2000 Wage Penalty, Replication",
                subtitle="Outsourced Coefficient on Log Wage, Row 1 (Author's Replication vs. Reported D&K)")
    .tab_spanner(label="Author's Replication", columns=["author", "author_star", "author_n"])
    .tab_spanner(label="D&K (2010), Table 3", columns=["dk", "dk_star", "dk_n"])
    .cols_label(author="Coef", author_n="N", dk="Coef", dk_n="N")
    .tab_source_note(source_note=(
        "Data: CPS-ORG via IPUMS for Author's Replication (1983-2000). D&K values are the reported row-1 estimates, "
        "as published (not CPS-ORG). Weights: EARNWT (replication choice; D&K do not state weighting). FE: state x year. "
        "Clustering: survey month (ym). Stars: * p<.10, ** p<.05, *** p<.01, two-tailed; D&K stars computed from the "
        "published SE. " + DEF_REPLICATION + " " + DEF_DK))
)
gt3 = style_regression(gt3, est_cols=["author", "dk"], se_rows=se_rows3, star_cols=["author_star", "dk_star"])
save_outputs(gt3, raw3, "tableA3_dk_pooled_replication")

In [ ]:
period_rows = [("2001-2003", "2001-2003"), ("2004-2006", "2004-2006"), ("2007-2009", "2007-2009"),
               ("2010-2012", "2010-2012"), ("2013-2015", "2013-2015"), ("2016-2019", "2016-2019"),
               ("2021-2025", "2021-2025"), ("Pooled 2001-2025", "2001-2025")]
occs = ["janitor", "guard"]

entries, raw_rows = [], []
for label, sample in period_rows:
    e = {"label": label, "est": {}, "single": {}}
    rrow = {"period": label}
    for occ in occs:
        r = get_row(estimates, spec="author", model="base", occ=occ, sample=sample, term="outsourced")
        e["est"][occ] = (r["estimate"], r["se"], r["p"])
        e["single"][occ + "_n"] = f"{int(r['n_obs']):,}"
        rrow[f"{occ}_estimate"] = r["estimate"]
        rrow[f"{occ}_se"] = r["se"]
        rrow[f"{occ}_p"] = r["p"]
        rrow[f"{occ}_n"] = int(r["n_obs"])
    entries.append(e)
    raw_rows.append(rrow)

single_cols = [occ + "_n" for occ in occs]
rows, se_rows = stacked_rows(entries, "period", occs, single_cols)
disp = stacked_frame(rows, "period", occs, single_cols)
disp = disp[["period", "janitor", "janitor_star", "janitor_n", "guard", "guard_star", "guard_n"]]
raw = pd.DataFrame(raw_rows)
gt4 = style(
    GT(disp, rowname_col="period")
    .tab_header(title="Table 3: Author's Extension of the Outsourcing Wage Penalty by Period, 2001-2025",
                subtitle="Outsourced Coefficient on Log Wage, Janitors and Guards; SE in Parentheses")
    .tab_spanner(label="Janitors", columns=["janitor", "janitor_star", "janitor_n"])
    .tab_spanner(label="Guards", columns=["guard", "guard_star", "guard_n"])
    .cols_label(janitor="Coef", janitor_n="N", guard="Coef", guard_n="N")
    .tab_source_note(source_note=(
        "Data: CPS-ORG via IPUMS. Author's Extension: the author's wage regression estimated separately for each "
        "period, 2001-2025 (2020 excluded). Weights: EARNWT. FE: state x year. Clustering: survey month (ym). "
        "Stars: * p<.10, ** p<.05, *** p<.01, two-tailed. No reported comparison values exist for these periods."))
)
gt4 = style_regression(gt4, est_cols=occs, se_rows=se_rows, star_cols=[occ + "_star" for occ in occs])
save_outputs(gt4, raw, "table3_penalty_2001_2025")

## Table 5 (body): minimum wage interaction, leads, and trend, 2001-2025

Built from `outputs/py/minwage_results.csv` (`05_minwage.ipynb`); no re-estimation.

In [ ]:
mw = pd.read_csv(f"{OUT}/py/minwage_results.csv")

def mw_row(occ_label, samp, model, term=None):
    m = mw[(mw["occ"] == occ_label) & (mw["sample"] == samp) & (mw["model"] == model)]
    m = m[m["term"].isna()] if term is None else m[m["term"] == term]
    assert len(m) == 1, (occ_label, samp, model, term, len(m))
    return m.iloc[0]

rows5, raw5 = [], []
for occ_label in ["Janitors", "Guards"]:
    i_ = mw_row(occ_label, "2001-2025", "interaction", "os_x_logmw")
    w = mw_row(occ_label, "2001-2025", "leads", None)
    tn = mw_row(occ_label, "2001-2025", "trend_without_mw", "os_x_yearc")
    tw = mw_row(occ_label, "2001-2025", "trend_with_mw", "os_x_yearc")
    rows5.append({
        "occupation": occ_label,
        "interaction": f"{i_['estimate']:.3f}{stars(i_['p'])} ({i_['se']:.3f}) [{i_['se_state']:.3f}]",
        "leads_test": f"chi2 = {w['stat']:.2f}, p = {w['p']:.3f}",
        "trend_without": f"{tn['estimate']:+.3f}{stars(tn['p'])} ({tn['se']:.3f})",
        "trend_with": f"{tw['estimate']:+.3f}{stars(tw['p'])} ({tw['se']:.3f})",
    })
    raw5.append({
        "occupation": occ_label, "interaction_estimate": i_["estimate"], "interaction_se": i_["se"],
        "interaction_se_state": i_["se_state"], "interaction_p": i_["p"], "interaction_p_state": i_["p_state"],
        "leads_wald_chi2": w["stat"], "leads_wald_df": w["df"], "leads_wald_p": w["p"],
        "trend_without_logmw": tn["estimate"], "trend_without_logmw_se": tn["se"], "trend_without_logmw_p": tn["p"],
        "trend_with_logmw": tw["estimate"], "trend_with_logmw_se": tw["se"], "trend_with_logmw_p": tw["p"],
    })
disp5 = pd.DataFrame(rows5)
raw5 = pd.DataFrame(raw5)
gt5 = style(
    GT(disp5, rowname_col="occupation")
    .tab_header(title="Table 5: Minimum Wage Interaction, Leads, and Trend, 2001-2025",
                subtitle="Outsourced x Log Minimum Wage; Clustered SE in Parentheses, State-Clustered SE in Brackets")
    .cols_label(interaction="Outsourced x log(MW)", leads_test="Joint Test on Leads",
                trend_without="Trend per Decade, Without log(MW)", trend_with="Trend per Decade, With log(MW)")
    .tab_source_note(source_note=(
        "Data: CPS-ORG via IPUMS for the analysis sample; minimum-wage source: data/minimumwage_annual.csv (FRED). "
        "From `05_minwage.ipynb`'s interaction model (outsourced x log_mw, FE STATEFIP^YEAR + outsourced^STATEFIP + "
        "outsourced^YEAR) and trend model (outsourced x year, centered 2001, same FE minus outsourced^YEAR), both "
        "with and without the log_mw interaction. Leads: joint Wald test (chi2, df 2) that the log_mw interactions "
        "at t+1 and t+2 are zero. Trend per decade = 10 x the outsourced x year coefficient. Clustering: survey "
        "month (ym); state-clustered SE in brackets for the interaction only. Stars from the ym-clustered p-value: "
        "* p<.10, ** p<.05, *** p<.01, two-tailed. Descriptive associations, not causal."))
)
save_outputs(gt5, raw5, "table5_minwage_2001_2025")

## R-squared for A1/A2 (one-off refit)

Refits only the four pooled author base models (janitors/guards x 1983-2000/2001-2025) to get R-squared and within R-squared, which are not in `estimates.csv`. Confirms the outsourced coefficient and SE match `estimates.csv` exactly before using the R-squared values.

In [ ]:
import pyfixest as pf

AUTHOR_FORMULA = (
    "log_wage ~ outsourced + union + parttime + educ_cat + female + black + hispanic "
    "+ age + age2 + msa + central_city + cc_unknown + geo_unidentified | STATEFIP^YEAR"
)
r2_cols = ["occupation", "YEAR", "STATEFIP", "EARNWT", "ym", "replication_sample", "extension_sample"] + MODEL_COV
r2_data = pd.read_parquet(f"{OUT}/py/dk_clean.parquet", columns=list(dict.fromkeys(r2_cols)))

R2 = {}  # (occ, sample) -> (r2, r2_within)
for occ in ["janitor", "guard"]:
    for sample, mask in [("1983-2000", r2_data["replication_sample"]), ("2001-2025", r2_data["extension_sample"])]:
        s = r2_data[(r2_data["occupation"] == occ) & mask].dropna(subset=MODEL_COV)
        fit = pf.feols(AUTHOR_FORMULA, data=s, weights="EARNWT", vcov={"CRV1": "ym"})
        r = fit.tidy().loc["outsourced"]
        bench_row = get_row(estimates, spec="author", model="base", occ=occ, sample=sample, term="outsourced")
        assert abs(r["Estimate"] - bench_row["estimate"]) < 1e-9, (occ, sample, "outsourced estimate mismatch")
        assert abs(r["Std. Error"] - bench_row["se"]) < 1e-9, (occ, sample, "outsourced SE mismatch")
        assert fit._N == int(bench_row["n_obs"]), (occ, sample, "N mismatch")
        R2[(occ, sample)] = (fit._r2, fit._r2_within)
        print(f"{occ} {sample}: R2={fit._r2:.4f} within R2={fit._r2_within:.4f} (coefficients confirmed to match estimates.csv)")

In [ ]:
term_order = ["outsourced", "union", "parttime",
              "educ_cat[T.no_school]", "educ_cat[T.primary]", "educ_cat[T.hs_attend]",
              "educ_cat[T.hs_complete]", "educ_cat[T.some_college]",
              "female", "black", "hispanic", "age", "age2",
              "msa", "central_city", "cc_unknown", "geo_unidentified"]
HIDDEN_TERMS = {"cc_unknown", "geo_unidentified"}  # estimated in the model; row omitted from the table
DK_TERMS = {"outsourced", "union", "parttime"}  # the only terms D&K's row 1 reports
term_label = {
    "outsourced": "Outsourced", "union": "Union", "parttime": "Part-time",
    "educ_cat[T.no_school]": "Education: no schooling (ref.: college)",
    "educ_cat[T.primary]": "Education: primary (ref.: college)",
    "educ_cat[T.hs_attend]": "Education: some high school (ref.: college)",
    "educ_cat[T.hs_complete]": "Education: high school completion (ref.: college)",
    "educ_cat[T.some_college]": "Education: some college (ref.: college)",
    "female": "Female", "black": "Black", "hispanic": "Hispanic", "age": "Age", "age2": "Age squared",
    "msa": "MSA", "central_city": "Central city", "cc_unknown": "MSA, central-city status unknown",
    "geo_unidentified": "Geography unidentified",
}
pooled_cols = [("dk", "dk"), ("1983-2000", "rep"), ("2001-2025", "ext")]
pooled_est = [col for _, col in pooled_cols]
pooled_star = [col + "_star" for col in pooled_est]
DATA_COLS = [c for col in pooled_est for c in (col, col + "_star")]
EXTRAP_COLS = ["ext"]  # 2001-2025 column, right of the 2000 break
TEX_RULES = [DATA_COLS.index(c) for c in EXTRAP_COLS]

def dk_row1(occ_label, term):
    r = bench[(bench["occ"] == occ_label) & (bench["row"] == 1) & (bench["term"] == term)]
    if len(r) == 0:
        return None
    r = r.iloc[0]
    return (r["estimate"], r["se"], p_two_sided(r["estimate"] / r["se"]))

def appendix_table(occ, occ_label, table_no):
    base = estimates[(estimates["occ"] == occ) & (estimates["model"] == "base") & (estimates["spec"] == "author")]
    unlabeled = set(base["term"]) - set(term_order) - HIDDEN_TERMS
    assert not unlabeled, f"unlabeled terms: {unlabeled}"
    entries, raw_rows = [], []
    for term in [t for t in term_order if t not in HIDDEN_TERMS]:
        e, rrow = {"label": term_label[term], "est": {}, "single": {}}, {"term": term}
        e["est"]["dk"] = dk_row1(occ_label, term) if term in DK_TERMS else None
        rrow["dk"] = e["est"]["dk"][0] if e["est"]["dk"] else np.nan
        for sample, col in pooled_cols[1:]:
            r = base[(base["sample"] == sample) & (base["term"] == term)]
            if len(r) == 0:
                e["est"][col] = None
                rrow[col] = np.nan
            else:
                r = r.iloc[0]
                e["est"][col] = (r["estimate"], r["se"], r["p"])
                rrow[col] = r["estimate"]
        entries.append(e)
        raw_rows.append(rrow)
    rows, se_rows = stacked_rows(entries, "term", pooled_est, [])

    # Observations (N) and D&K's reported R-squared (not computed here for the author's columns: no re-estimation)
    nrow, rnrow = {"term": "Observations (N)"}, {"term": "Observations (N)"}
    dk_n = bench[(bench["occ"] == occ_label) & (bench["row"] == 1) & (bench["term"] == "outsourced")]["n"].iloc[0]
    nrow["dk"], nrow["dk_star"], rnrow["dk"] = f"{int(dk_n):,}", "", int(dk_n)
    for sample, col in pooled_cols[1:]:
        n = int(base[base["sample"] == sample]["n_obs"].iloc[0])
        nrow[col], nrow[col + "_star"], rnrow[col] = f"{n:,}", "", n
    r2row, rr2row = {"term": "R-squared"}, {"term": "R-squared"}
    wr2row, rwr2row = {"term": "Within R-squared"}, {"term": "Within R-squared"}
    dk_r2 = bench[(bench["occ"] == occ_label) & (bench["row"] == 1) & (bench["term"] == "outsourced")]["r2"].iloc[0]
    r2row["dk"], r2row["dk_star"], rr2row["dk"] = f"{dk_r2:.2f}", "", dk_r2
    wr2row["dk"], wr2row["dk_star"], rwr2row["dk"] = "", "", np.nan  # D&K do not report within R-squared
    for sample, col in pooled_cols[1:]:
        r2_val, r2w_val = R2[(occ, sample)]
        r2row[col], r2row[col + "_star"], rr2row[col] = f"{r2_val:.2f}", "", r2_val
        wr2row[col], wr2row[col + "_star"], rwr2row[col] = f"{r2w_val:.2f}", "", r2w_val
    rows += [nrow, r2row, wr2row]
    raw_rows += [rnrow, rr2row, rwr2row]

    disp = stacked_frame(rows, "term", pooled_est, [])
    raw = pd.DataFrame(raw_rows)
    gt_obj = style(
        GT(disp, rowname_col="term")
        .tab_header(title=f"Table A{table_no}: Pooled Wage Regression Coefficients, {occ_label}",
                    subtitle="Dependent Variable: Log Hourly Wage. Coefficients With SE in Parentheses")
        .tab_spanner(label="D&K, 1983-2000", columns=["dk", "dk_star"])
        .tab_spanner(label="Author's Replication", columns=["rep", "rep_star"])
        .tab_spanner(label="Author's Extension", columns=["ext", "ext_star"])
        .cols_label(dk="Coef", rep="1983-2000", ext="2001-2025")
        .tab_source_note(source_note=(
            "Data: CPS-ORG via IPUMS for the author's columns. D&K, 1983-2000: Dube and Kaplan's (2010) row-1 "
            "published coefficients, as published (not CPS-ORG); D&K report only outsourced, union, and parttime, so "
            "the other term rows and the R-squared are blank for that column (not reported in the paper). The "
            "author's R-squared and within R-squared come from a one-off refit of the four pooled base models ""(janitors/guards x 1983-2000/2001-2025), confirmed to match estimates.csv's outsourced coefficient, SE, ""and N exactly; D\\&K do not report within R-squared. Author's Replication: 1983-2000, the "
            "author's wage regression. Author's Extension: 2001-2025, estimated on its own data. " + EXTRAP_RULE_DEF + " "
            "Weights: EARNWT. FE: state x year (STATEFIP^YEAR), not shown as rows. Clustering: survey month (ym), CRV1 "
            "(D&K's clustering is as published). Reference education: college. Part-time = usual hours under 30. "
            "Blank cells: term not in that model or not reported. The MSA central-city-status-unknown and "
            "geography-unidentified indicators are in the model but not shown. Stars: * p<.10, ** p<.05, *** p<.01, "
            "two-tailed; D&K stars computed here from the published SE. Table A3 gives the full D&K vs. author "
            "comparison for the outsourced coefficient."))
    )
    gt_obj = style_regression(gt_obj, est_cols=pooled_est, se_rows=se_rows, star_cols=pooled_star,
                              extrap_cols=EXTRAP_COLS)
    return gt_obj, raw

gta1, rawa1 = appendix_table("janitor", "Janitors", "1")
save_outputs(gta1, rawa1, "tableA1_janitors_pooled_coefficients", tex_rules=TEX_RULES)
gta2, rawa2 = appendix_table("guard", "Guards", "2")
save_outputs(gta2, rawa2, "tableA2_guards_pooled_coefficients", tex_rules=TEX_RULES)

## Regenerate outputs/RESULTS.md

Built from the same CSVs as the tables.

In [ ]:
t3 = pd.read_csv(f"{OUT}/tables/tableA3_dk_pooled_replication.csv")
t4 = pd.read_csv(f"{OUT}/tables/table3_penalty_2001_2025.csv")

def pooled_line(df, occ_name, occ):
    r = df[df["period"] == "Pooled 2001-2025"].iloc[0]
    return f"- {occ_name}: {r[occ + '_estimate']:.3f} (SE {r[occ + '_se']:.3f}); N = {int(r[occ + '_n']):,}"

lines = [
    "# Results: outsourcing wage penalty replication",
    "",
    "Regenerated by `notebooks/03_tables.ipynb` from `outputs/py/`. Do not edit by hand. Exhibit numbering follows "
    "CLAUDE.md's Paper structure: body is 2001-2025 only (no D&K comparisons except the incidence figure); the "
    "appendix holds the 1983-2000 validation against D&K and any 1983-2025 series.",
    "",
    "## Body",
    "- Table 1: `table1_janitors_workforce`: janitors, workforce characteristics, 2001-2025.",
    "- Table 2: `table2_guards_workforce`: guards, same layout.",
    "- Table 3: `table3_penalty_2001_2025`: outsourcing penalty by period, 2001-2025 plus pooled, janitors and guards. No D&K.",
    "- Table 4: `table4_union_mediation` (built by `06_unionmediation.ipynb`): union mediation, 5-year bins, 2001-2005 through 2021-2025.",
    "- Table 5: `table5_minwage_2001_2025`: minimum wage interaction, leads, and trend, 2001-2025.",
    "- Figure 1: `figures/figure1_incidence_annual.png`: annual outsourcing incidence, with D&K Table 1 period averages for comparison.",
    "- Figure 2: `figures/figure2_penalty_by_period_2001_2025.png` (built by `02_estimates.ipynb`): outsourcing penalty by period, 2001-2025 only.",
    "- Figure 3: `figures/figure3_minwage_mechanism_2001_2025.png` (built by `05_minwage.ipynb`): share earning below 1.1 x binding minimum, 2001-2025.",
    "",
    "## Appendix",
    "- Table A1: `tableA1_janitors_pooled_coefficients`: janitors, all coefficients; D&K 1983-2000 (row 1), Author's Replication (1983-2000), Author's Extension (2001-2025). R-squared and within R-squared from a one-off refit of the four pooled base models.",
    "- Table A2: `tableA2_guards_pooled_coefficients`: guards, same layout.",
    "- Table A3: `tableA3_dk_pooled_replication`: Author's Replication (1983-2000) vs. D&K reported values, outsourced coefficient.",
    "- Table A4: `tableA4_robustness`: robustness variants, outsourced coefficient (SE) and N, panels for 2001-2025 and 1983-2000.",
    "- Table A5: `tableA5_union_premium_interactions` (built by `06_unionmediation.ipynb`): union x outsourced interactions, panels for 2001-2025 and 1983-2000 bins.",
    "- Table A6: `tableA6_union_mediation_1983_2000` (built by `06_unionmediation.ipynb`): union mediation, 1983-1987 through 1998-2000, same sign convention as Table 4.",
    "- Figure A: `figures/figureA_penalty_by_period_1983_2025.png` (built by `02_estimates.ipynb`): outsourcing penalty by period, full 1983-2025 range.",
    "- Figure B: `figures/figureB_penalty_dk_vs_replication.png` (built by `02_estimates.ipynb`): D&K published penalty vs. Author's Replication, pooled 1983-2000.",
    "- Long-run trend tests (1983-2025 contrasts) and the quarterly 2000-2002 estimates: `outputs/RESULTS_trends.md`, Appendix section.",
    "- 1983-2025 minimum wage results: `outputs/RESULTS_minwage.md`, Appendix section.",
    "- PNG: not generated for HTML tables (needs Selenium; Chrome is installed).",
    "",
    "## Headline numbers",
    "Table A3, pooled 1983-2000, outsourced coefficient (Author's Replication vs. D&K reported row 1):",
]
for _, r in t3.iterrows():
    lines.append(f"- {r['occupation']}: Author's Replication {r['author_estimate']:.3f} (SE {r['author_se']:.3f}, N {int(r['author_n']):,}); "
                 f"D&K reported {r['dk_estimate']:.3f} (SE {r['dk_se']:.3f}, N {int(r['dk_n']):,})")
lines += ["", "Pooled 2001-2025, Author's Extension (body, Table 3):", pooled_line(t4, "Janitors", "janitor"),
          pooled_line(t4, "Guards", "guard"), "",
          "## Caveats",
          "- Replication N is above D&K's published N for both occupations (Table A3). The cause is not established.",
          "- Tables 1-2 use the author's regression sample, so they can differ slightly from the per-variable summary statistics in 02_estimates section 5.",
          "- Workers represented are estimated from EARNWT on all base-sample workers, not only the regression sample.",
          "- Janitor rows 2-3 (sex split) are not in these tables, so the transposed-label question does not arise here.",
          "- The author's model is the D&K row-1 wage regression without gov_employee. gov_employee appears only in the descriptive Tables 1-2 and the section 5 summary statistics.",
          ""]
with open(f"{OUT}/RESULTS.md", "w") as f:
    f.write("\n".join(lines))
print("wrote outputs/RESULTS.md")

## Table A4: Robustness variants

One row per variant: outsourced coefficient (SE) and N, janitors and guards, 1983-2000 and 2001-2025. The main estimate is the first row.

In [ ]:
# Table A4: robustness variants (spec = author for the main estimate, author_<check> otherwise)
est = estimates
VARIANTS = [("author", "Main estimate (part-time under 30 hours)"),
            ("author_parttime35", "Part-time under 35 hours"),
            ("author_gov", "Including gov_employee"),
            ("author_untrimmed", "Untrimmed wages"),
            ("author_unweighted", "Unweighted"),
            ("author_completecase_geo", "Complete cases on geography"),
            ("author_hourly_only", "Hourly-paid workers only"),
            ("author_no_localgov", "Excluding local government")]
PANELS4 = [("2001-2025", "2001-2025"), ("1983-2000", "1983-2000")]

def pick_variant(spec, occ, samp):
    r = est[(est["spec"] == spec) & (est["model"] == "base") & (est["term"] == "outsourced")
            & (est["occ"] == occ) & (est["sample"] == samp)]
    assert len(r) == 1, (spec, occ, samp, len(r))
    return r.iloc[0]

entries4, raw4 = [], []
for panel_label, samp in PANELS4:
    for spec, label in VARIANTS:
        e = {"group": panel_label, "label": label, "est": {}, "single": {}}
        rrow = {"panel": panel_label, "variant": label, "spec": spec}
        for occ, col in [("janitor", "jan"), ("guard", "gd")]:
            r = pick_variant(spec, occ, samp)
            e["est"][col] = (r["estimate"], r["se"], r["p"])
            e["single"][col + "_n"] = f"{int(r['n_obs']):,}"
            rrow.update({f"{col}_estimate": r["estimate"], f"{col}_se": r["se"], f"{col}_p": r["p"],
                        f"{col}_n": int(r["n_obs"])})
        entries4.append(e)
        raw4.append(rrow)

est4 = ["jan", "gd"]
single4 = [col + "_n" for col in est4]
rows4, se_rows4 = stacked_rows(entries4, "variant", est4, single4, group_key="panel")
disp4 = stacked_frame(rows4, "variant", est4, single4, group_key="panel")
disp4 = disp4[["panel", "variant", "jan", "jan_star", "jan_n", "gd", "gd_star", "gd_n"]]
raw4 = pd.DataFrame(raw4)
gt_a4 = style(
    GT(disp4, rowname_col="variant", groupname_col="panel")
    .tab_header(title="Table A4: Robustness of the Outsourcing Wage Penalty",
                subtitle="Outsourced Coefficient on Log Wage; SE in Parentheses; Pooled Samples")
    .tab_spanner(label="Janitors", columns=["jan", "jan_star", "jan_n"])
    .tab_spanner(label="Guards", columns=["gd", "gd_star", "gd_n"])
    .cols_label(jan="Coef", jan_n="N", gd="Coef", gd_n="N")
    .tab_source_note(source_note=(
        "Data: CPS-ORG via IPUMS. Each row is one variant of the author's wage regression (base model). Panels: "
        "2001-2025 (Author's Extension) and 1983-2000 (Author's Replication). Part-time = usual hours under 30 in "
        "the main estimate; the 35-hour variant uses the BLS threshold. Unweighted: no EARNWT weights. Untrimmed: "
        "log wage before the trimming rule. Complete cases on geography drop METRO 0, 4, 9 and the two geography "
        "indicators. Hourly-paid: PAIDHOUR == 2. Excluding local government: CLASSWKR 28 removed. N is the "
        "regression sample after pyfixest drops singleton fixed effects. Stars: * p<.10, ** p<.05, *** p<.01, "
        "two-tailed."))
)
gt_a4 = style_regression(gt_a4, est_cols=est4, se_rows=se_rows4, star_cols=[col + "_star" for col in est4])
save_outputs(gt_a4, raw4, "tableA4_robustness")